# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
env = dbutils.widgets.text('env','dev')

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
env = dbutils.widgets.get("env")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog}.{schema}.dim_customer
(
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,

    customer_id STRING,
    customer_name STRING,
    email STRING,
    city STRING,
    state STRING,

    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,

    created_ts TIMESTAMP,
    updated_ts TIMESTAMP
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/{schema}/dim_customer'
""")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog}.{schema}.dim_product
(
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,

    product_id STRING,
    product_name STRING,
    category STRING,
    brand STRING,

    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,

    created_ts TIMESTAMP,
    updated_ts TIMESTAMP
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/{schema}/dim_product'
""")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog}.{schema}.dim_date
(
    date_key INT,
    full_date DATE,

    year INT,
    quarter INT,

    month INT,
    month_name STRING,

    week_of_year INT,

    day_of_month INT,
    day_name STRING
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/{schema}/dim_date'
""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.{schema}.fact_sales
(
    sales_key BIGINT GENERATED ALWAYS AS IDENTITY,

    order_id STRING,

    customer_key BIGINT,
    product_key BIGINT,

    date_key INT,

    quantity INT,
    unit_price DECIMAL(10,2),

    sales_amount DECIMAL(18,2),

    created_ts TIMESTAMP
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/{schema}/fact_sales'
""")

In [0]:
spark.sql(
    f"""CREATE TABLE IF NOT EXISTS {catalog}.{schema}.fact_returns
(
    return_key BIGINT GENERATED ALWAYS AS IDENTITY,

    return_id STRING,

    product_key BIGINT,

    date_key INT,

    refund_amount DECIMAL(18,2),

    created_ts TIMESTAMP
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/{schema}/fact_returns'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.util.etl_control
(
    table_name STRING,
    last_processed_ts TIMESTAMP
)
USING DELTA
location 'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/util/etl_control'
""")